In [ ]:
import requests
api_key = "sk-ac80bb3b944f499db1229c51f5172e88"
# 1. 造一个天气工具的类
class WeatherTool:
    def __init__(self):
        # 给你的工具起个名字，这很重要，因为大模型要靠这个名字来调用它
        self.name = "get_weather"
        self.description = "查询南宁的真实天气情况"

    # 这就是工具的核心动作，大模型发指令后，代码就会执行这个函数
    def execute(self, args):
        # 请把你昨天写的那个请求 Open-Meteo 天气 API 的代码搬到这里来      
        # 提示：
        # 1. 定义 url
        # 2. 用 requests.get 发请求
        # 3. 提取温度
        # 4. return "南宁当前气温 wind = data["current_weather"]["windspeed"]XX 度"
        url = "https://api.open-meteo.com/v1/forecast?latitude=22.82&longitude=108.32&current_weather=true" 
        try: 
            res = requests.get(url, timeout=5)
            data = res.json()
            temp = data["current_weather"]["temperature"]
            wind = data["current_weather"]["windspeed"]
            return f"南宁当前气温 {temp}°C，风速 {wind} km/h。"
        except Exception as e:  
            return f"天气查询失败：{e}"

# ==========================================
# 测试你的黑盒
# ==========================================
weather = WeatherTool()
# 模拟大模型调用它，传个空字典作为参数
print(weather.execute({}))          

In [ ]:
import os
class KnowledgeTool:
    def __init__(self):
        self.name = "search_knowledge"
        self.decription = "查询本地知识库"
    def execute(self, args):
        query = args.get("query","")
        if not os.path.exists("knowledge.txt"):
            return "未找到知识库文件。"
        with open("knowledge.txt", "r", encoding="utf-8") as f:
            content = f.read()
        paragraphs = content.split("\n")   
        results = [p for p in paragraphs if any(query[i:i+2] in p for i in range(len(query) - 1))]
        if results:
            return "在知识库中找到了以下相关内容：\n" + "\n".join(results[:3])
        else:
            return "知识库中没有找到相关内容。"

                   

In [ ]:
# 实例化你的知识库工具
kb = KnowledgeTool()

# 模拟大模型发来指令，要查“迟到”
print(kb.execute({"query": "迟到"}))

In [ ]:
import requests
import json
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
import requests
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_deepseek import ChatDeepSeek
from langchain_huggingface import HuggingFaceEmbeddings

class KnowledgeTool:
    def __init__(self):
        self.name = "search_knowledge"
        self.description = "查询本地知识库"
    def execute(self, args):
        query = args.get("query","")
        if not os.path.exists("knowledge.txt"):
            return "未找到知识库文件。"
        with open("knowledge.txt", "r", encoding="utf-8") as f:
            content = f.read()
        paragraphs = content.split("\n")   
        results = [p for p in paragraphs if any(query[i:i+2] in p for i in range(len(query) - 1))]
        if results:
            return "在知识库中找到了以下相关内容：\n" + "\n".join(results[:3])
        else:
            return "知识库中没有找到相关内容。"
class AdvancedKnowledgeTool:
    def __init__(self, pdf_path, api_key):
        self.name = "search_pdf_document"
        self.description = "当你需要查询长篇文档（如PDF）中的具体信息时，请调用此工具，传入你要查的问题。"
        
        print("⏳ 正在初始化 RAG 引擎（真实向量版）...")
        self.llm = ChatDeepSeek(model="deepseek-chat", api_key=api_key, temperature=0)
        
        # 🌟 下载并加载真实的向量模型（all-MiniLM-L6-v2 是业界极轻量极好用的模型）
        self.embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
        
        loader = PyPDFLoader(pdf_path)
        docs = loader.load()
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
        self.splits = text_splitter.split_documents(docs)
        
        # 用真实的 embeddings 来建库
        self.vectorstore = Chroma.from_documents(self.splits, self.embeddings)
        print(f"✅ RAG 引擎初始化完毕！共载入 {len(self.splits)} 个文本块。")

    def execute(self, args):
        query = args.get("query", "")
        if not query:
            return "未提供查询内容。"
            
        # 真正去向量库里搜索
        docs = self.vectorstore.similarity_search(query, k=3)
        context = "\n\n".join([doc.page_content for doc in docs])
        
        prompt = f"""
        请仅根据以下提供的上下文，用最简练的语言回答用户的问题。如果上下文没有提到，请直接说“不知道”。
        【上下文】：{context}
        【用户问题】：{query}
        """
        try:
            response = self.llm.invoke(prompt)
            return response.content
        except Exception as e:
            return f"查询失败：{e}"
            
class WeatherTool:
    def __init__(self):
        # 给你的工具起个名字，这很重要，因为大模型要靠这个名字来调用它
        self.name = "get_weather"
        self.description = "查询南宁的真实天气情况"

    # 这就是工具的核心动作，大模型发指令后，代码就会执行这个函数
    def execute(self, args):
        # 请把你昨天写的那个请求 Open-Meteo 天气 API 的代码搬到这里来      
        # 提示：
        # 1. 定义 url
        # 2. 用 requests.get 发请求
        # 3. 提取温度
        # 4. return "南宁当前气温 wind = data["current_weather"]["windspeed"]XX 度"
        url = "https://api.open-meteo.com/v1/forecast?latitude=22.82&longitude=108.32&current_weather=true" 
        try: 
            res = requests.get(url, timeout=5)
            data = res.json()
            temp = data["current_weather"]["temperature"]
            wind = data["current_weather"]["windspeed"]
            return f"南宁当前气温 {temp}°C，风速 {wind} km/h。"
        except Exception as e:  
            return f"天气查询失败：{e}"

class SuperAgent:
    def __init__(self, api_key):
        self.api_key = api_key
        self.url = "https://api.deepseek.com/chat/completions"
        
        # 🌟 把咱们造好的两个黑盒装进“工具箱”里！
        self.tools = [
            WeatherTool(),
            AdvancedKnowledgeTool("my_doc.pdf", api_key)
        ]

    # 动态生成给大模型的“工具说明书”
    def get_tools_description(self):
        desc = "你可以使用以下工具。如果需要使用，请仅输出 JSON 格式：{\"tool\": \"工具名\", \"args\": {}}\n"
        for t in self.tools:
            desc += f"- {t.name}: {t.description}\n"
        return desc

    # 根据名字精准找到黑盒并执行
    def execute_tool(self, tool_name, args):
        for t in self.tools:
            if t.name == tool_name:
                return t.execute(args)
        return "未找到该工具"

    # 主脑逻辑：和昨天一模一样，但极度精简
    def chat(self, user_message):
        system_prompt = f"""你是一个智能助手。{self.get_tools_description()}
【绝对铁律】：当用户询问天气、特定文档或规定时，你【必须】使用工具。
如果你决定使用工具，请【仅输出】如下格式的 JSON，绝对不要输出任何其他废话：
{{"tool": "工具名", "args": {{"query": "用户的问题"}}}}
"""
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ]
        
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json; charset=utf-8"
        }

        # 第一次请求
        data = {"model": "deepseek-chat", "messages": messages}
        response = requests.post(self.url, headers=headers, data=json.dumps(data, ensure_ascii=False).encode('utf-8'))
        ai_reply = response.json()["choices"][0]["message"]["content"]

        # 判断是否调用工具
        if "{" in ai_reply and "tool" in ai_reply:
            start = ai_reply.find("{")
            end = ai_reply.rfind("}") + 1
            try:
                tool_json = json.loads(ai_reply[start:end])
                tool_name = tool_json.get("tool")
                args = tool_json.get("args", {})
                
                # 执行黑盒
                tool_result = self.execute_tool(tool_name, args)
                print(f"🔧 [系统底层] 调用了工具：{tool_name}，结果：{tool_result}")
                
                # 第二次请求
                messages.append({"role": "assistant", "content": ai_reply})
                messages.append({"role": "user", "content": f"工具结果：{tool_result}。请根据此回答用户。"})
                
                data = {"model": "deepseek-chat", "messages": messages}
                final_response = requests.post(self.url, headers=headers, data=json.dumps(data, ensure_ascii=False).encode('utf-8'))
                return final_response.json()["choices"][0]["message"]["content"]
            except Exception as e:
                return f"解析工具失败：{e}"
        else:
            return ai_reply

# ==========================================
# 测试终极 Agent
# ==========================================
my_key = ""
agent = SuperAgent(my_key)

print(agent.chat("南宁今天天气怎么样？"))
print(agent.chat("代表性传承人级别是什么？"))